# Notebook 18b — NSV Stage 1 on Lowpassed Data (BSISO MJJAS, lp25)
**Project:** ENSO-BSISO SSL — Neural State Variables extension  
**Author:** Jiayi (jh9141@nyu.edu)

Variant of `nb18` that consumes the **lp25** consecutive-day pairs from `nb17b` (synoptic noise removed by the 25-day Lanczos lowpass). Same encoder, same decoder, same hyperparameters, same training loop. Only the input data differs.

## Why this variant exists

`nb18` (run on Lee-only pairs) produced a 64-D latent whose ID estimate (`nb19`) came back at **d̂ ≈ 17 with LOW confidence** — the Gaussian-noise calibration failed (saturated at 39.4 instead of 64), the PCA scree was flat (PC1 only 6.5%), and the latent showed no BSISO phase or ENSO organization in the PC1×PC2 plane. The encoder absorbed synoptic-eddy variance instead of the BSISO state.

Feeding the encoder lowpassed input (`X_MJJAS_lee_lp25.npy`) should fix this: signals faster than 25 d are removed, so the network can only learn from the slow intraseasonal modes that BSISO actually occupies. Same encoder, same N, same hyperparameters — clean ablation.

## Architecture

**Unchanged from nb18.** Encoder is 5 conv blocks → 64-D bottleneck; decoder is bilinear-upsample + conv pyramid → `(3, 31, 51)`. Total ~230 K params. See nb18 header for the layer-by-layer table.

## Training

Same hyperparameters as nb18 (Adam, lr=1e-3, CosineAnnealingLR, 100 epochs, batch 64, weight decay 1e-4, MSE loss). On the smoother lowpassed data we expect:
- Lower final train + val MSE than nb18 (predictable signal is larger fraction of total variance).
- Bigger improvement over persistence (slow state is more autocorrelated → both persistence and the model do better, but the model gains more).
- Latent spread out **less** isotropically — the manifold should be visibly low-D in PCA.

## Inputs (from nb17b)

- `nsv/data_lp25/X_t.npy`, `X_t1.npy` — lp25 pairs, shape `(~6,400, 3, 31, 51)`
- `nsv/data_lp25/train_mask.npy` — same year-based split
- ancillary labels — pass-through

## Outputs (`BSISO_SSL_Project/nsv/`)

- `checkpoints_lp25/encoder_stage1.pth`, `decoder_stage1.pth`, `*_best.pth`
- `checkpoints_lp25/training_history_stage1.json`
- `latents_lp25/z_train.npy`, `z_val.npy` ← these are what nb19 will re-run on
- `results/stage1_lp25/training_curves.png`, `reconstructions.png`, `latent_diagnostics.png`, `stage1_summary.json`

## Verification gate

1. Beats persistence by a **wider margin** than nb18 (lp25 is more predictable).
2. Reconstructions visually match the lp25 targets (smooth large-scale patterns).
3. Latent non-degenerate (≥ 16 dims active) — but on lp25 we *also* hope the variance becomes **non-isotropic** (PC1 ≫ PC2 ≫ ... rather than flat). That's the key Stage 1 → Stage 2 sanity.

## Runtime

~3 min on Colab T4 (same model size as nb18, same N).

---

## Cell 1 — Mount Drive, Load nb17b Outputs, Setup

Only path changes vs nb18 Cell 1: `DATA_DIR`, `CKPT_DIR`, `LATENT_DIR`, `RESULTS_DIR` all suffixed `_lp25`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

PROJECT_DIR  = '/content/drive/MyDrive/BSISO_SSL_Project'
NSV_DIR      = f'{PROJECT_DIR}/nsv'
DATA_DIR     = f'{NSV_DIR}/data_lp25'
CKPT_DIR     = f'{NSV_DIR}/checkpoints_lp25'
LATENT_DIR   = f'{NSV_DIR}/latents_lp25'
RESULTS_DIR  = f'{NSV_DIR}/results/stage1_lp25'
for d in [CKPT_DIR, LATENT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

LATENT_DIM   = 64
BATCH_SIZE   = 64
EPOCHS       = 100
LR           = 1e-3
WEIGHT_DECAY = 1e-4
SEED         = 42

torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

X_t        = np.load(f'{DATA_DIR}/X_t.npy')
X_t1       = np.load(f'{DATA_DIR}/X_t1.npy')
train_mask = np.load(f'{DATA_DIR}/train_mask.npy')
with open(f'{DATA_DIR}/nsv_data_meta.json') as f:
    meta = json.load(f)

n_total = X_t.shape[0]
n_train = int(train_mask.sum())
n_val   = int((~train_mask).sum())
assert X_t.shape == X_t1.shape == (n_total, 3, 31, 51), f'Unexpected pair shape {X_t.shape}'
print(f'Loaded {n_total} lp25 pairs from nb17b  (train {n_train} / val {n_val}).')
print(f'X_t range: [{X_t.min():.3f}, {X_t.max():.3f}],  std={X_t.std():.3f}  '
      f'(lp25 std is lower than Lee-only because synoptic energy removed)')

## Cell 2 — Encoder + Decoder (Identical to nb18)

Same `EncoderBSISO` and `DecoderBSISO` classes from nb18 — copied so this notebook is self-contained. Architecture decisions justified in nb18 Cell 2 header.

In [ ]:
class EncoderBSISO(nn.Module):
    """CNN encoder: (B, 3, 31, 51) -> (B, latent_dim)."""
    def __init__(self, latent_dim=64):
        super().__init__()
        self.conv1 = nn.Conv2d(3,  32,  kernel_size=4, stride=2, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 32,  kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64,  kernel_size=4, stride=2, padding=1, bias=False)
        self.bn3   = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 64,  kernel_size=3, stride=1, padding=1, bias=False)
        self.bn4   = nn.BatchNorm2d(64)
        self.conv5 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False)
        self.bn5   = nn.BatchNorm2d(128)
        self.gap   = nn.AdaptiveAvgPool2d(1)
        self.fc    = nn.Linear(128, latent_dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01); nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = F.relu(self.bn5(self.conv5(x)))
        x = self.gap(x).flatten(1)
        return self.fc(x)


class DecoderBSISO(nn.Module):
    """Bilinear-upsample + conv decoder: (B, latent_dim) -> (B, 3, 31, 51)."""
    def __init__(self, latent_dim=64):
        super().__init__()
        self.fc    = nn.Linear(latent_dim, 128)
        self.conv1 = nn.Conv2d(128, 64, kernel_size=3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64,  64, kernel_size=3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64,  32, kernel_size=3, padding=1, bias=False)
        self.bn3   = nn.BatchNorm2d(32)
        self.conv4 = nn.Conv2d(32,  3,  kernel_size=3, padding=1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01); nn.init.constant_(m.bias, 0)

    def forward(self, z):
        x = self.fc(z).view(-1, 128, 1, 1)
        x = F.interpolate(x, size=(3, 6),   mode='bilinear', align_corners=False)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.interpolate(x, size=(7, 12),  mode='bilinear', align_corners=False)
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.interpolate(x, size=(15, 25), mode='bilinear', align_corners=False)
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.interpolate(x, size=(31, 51), mode='bilinear', align_corners=False)
        return self.conv4(x)


enc = EncoderBSISO(LATENT_DIM).to(device)
dec = DecoderBSISO(LATENT_DIM).to(device)
n_params = sum(p.numel() for p in enc.parameters()) + sum(p.numel() for p in dec.parameters())

with torch.no_grad():
    dummy = torch.randn(4, 3, 31, 51).to(device)
    z = enc(dummy); xhat = dec(z)
    assert z.shape == (4, LATENT_DIM); assert xhat.shape == dummy.shape
    print(f'Total params: {n_params:,}  (same as nb18, ~230K).  Forward sanity OK.')

## Cell 3 — Dataset + DataLoaders + Persistence Baseline

In [ ]:
class PairDataset(Dataset):
    def __init__(self, X_t, X_t1, indices):
        self.X_t  = torch.from_numpy(X_t[indices]).float()
        self.X_t1 = torch.from_numpy(X_t1[indices]).float()
    def __len__(self):  return self.X_t.shape[0]
    def __getitem__(self, k):  return self.X_t[k], self.X_t1[k]

train_idx = np.where(train_mask)[0]
val_idx   = np.where(~train_mask)[0]
train_ds = PairDataset(X_t, X_t1, train_idx)
val_ds   = PairDataset(X_t, X_t1, val_idx)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds):5d} pairs  -> {len(train_loader)} batches/epoch')
print(f'Val:   {len(val_ds):5d} pairs  -> {len(val_loader)} batches/epoch')

with torch.no_grad():
    persistence_mse = ((val_ds.X_t - val_ds.X_t1) ** 2).mean().item()
print(f'Persistence MSE on val (predict X_t+1 = X_t): {persistence_mse:.4f}')
print('  (Lee-only baseline was 1.231;  lp25 expected lower because smoother fields → higher day-to-day autocorrelation.)')

## Cell 4 — Training Loop (Identical to nb18)

In [ ]:
from tqdm.notebook import tqdm

params = list(enc.parameters()) + list(dec.parameters())
optimizer = optim.Adam(params, lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.01)

history = {'train_mse': [], 'val_mse': [], 'epoch_time': []}
best_val = float('inf')

for epoch in range(EPOCHS):
    t0 = time.time()

    enc.train(); dec.train()
    train_loss = 0.0; n_train_seen = 0
    pbar = tqdm(train_loader, desc=f'ep {epoch+1}/{EPOCHS}', leave=False)
    for x_t, x_t1 in pbar:
        x_t  = x_t.to(device, non_blocking=True)
        x_t1 = x_t1.to(device, non_blocking=True)
        loss = F.mse_loss(dec(enc(x_t)), x_t1)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        train_loss += loss.item() * x_t.size(0); n_train_seen += x_t.size(0)
        pbar.set_postfix({'mse': f'{loss.item():.4f}'})
    train_mse = train_loss / n_train_seen

    enc.eval(); dec.eval()
    val_loss = 0.0; n_val_seen = 0
    with torch.no_grad():
        for x_t, x_t1 in val_loader:
            x_t  = x_t.to(device, non_blocking=True)
            x_t1 = x_t1.to(device, non_blocking=True)
            val_loss += F.mse_loss(dec(enc(x_t)), x_t1, reduction='sum').item() / x_t1.numel() * x_t.size(0)
            n_val_seen += x_t.size(0)
    val_mse = val_loss / n_val_seen
    scheduler.step()

    et = time.time() - t0
    history['train_mse'].append(train_mse); history['val_mse'].append(val_mse); history['epoch_time'].append(et)
    print(f'ep {epoch+1:3d}/{EPOCHS}  train={train_mse:.4f}  val={val_mse:.4f}  '
          f'lr={scheduler.get_last_lr()[0]:.2e}  time={et:.1f}s')

    if val_mse < best_val:
        best_val = val_mse
        torch.save(enc.state_dict(), f'{CKPT_DIR}/encoder_stage1_best.pth')
        torch.save(dec.state_dict(), f'{CKPT_DIR}/decoder_stage1_best.pth')

torch.save(enc.state_dict(), f'{CKPT_DIR}/encoder_stage1.pth')
torch.save(dec.state_dict(), f'{CKPT_DIR}/decoder_stage1.pth')
with open(f'{CKPT_DIR}/training_history_stage1.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f'\nDone in {sum(history["epoch_time"])/60:.1f} min.')
print(f'Best val MSE:    {best_val:.4f}  (at epoch {history["val_mse"].index(best_val)+1}).')
print(f'Final train MSE: {history["train_mse"][-1]:.4f}    val MSE: {history["val_mse"][-1]:.4f}')
print(f'Persistence:     {persistence_mse:.4f}')
ratio = (persistence_mse - best_val) / persistence_mse * 100
print(f'Improvement over persistence: {ratio:.1f}%  (nb18 Lee-only achieved 27.5%; expect larger here)')

## Cell 5 — Training Curves + Reconstruction + Latent Diagnostics

Same three diagnostics as nb18 Cell 5. The **most informative panel here** is the latent-diagnostics bar chart at the end: on lp25 we hope to see the per-dim std become **non-uniform** (a few dominant dims with much larger std, rest small) — this is the visual signature of a low-D manifold. The flat per-dim profile we saw on nb18 (all dims std ≈ 0.03) was the early warning that the latent didn't manifoldize.

In [ ]:
enc.load_state_dict(torch.load(f'{CKPT_DIR}/encoder_stage1_best.pth', map_location=device))
dec.load_state_dict(torch.load(f'{CKPT_DIR}/decoder_stage1_best.pth', map_location=device))
enc.eval(); dec.eval()

# 1) Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(history['train_mse'], label='Train MSE', lw=2)
axes[0].plot(history['val_mse'],   label='Val MSE',   lw=2)
axes[0].axhline(persistence_mse, color='gray', ls='--', lw=1, label=f'Persistence ({persistence_mse:.3f})')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE')
axes[0].set_title('Training Curves (lp25)', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history['epoch_time'], color='green', lw=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Time (s)')
axes[1].set_title('Per-Epoch Time', fontweight='bold')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_curves.png', dpi=140, bbox_inches='tight')
plt.show()

# 2) Reconstructions
rng = np.random.default_rng(SEED)
picks = rng.choice(len(val_ds), size=4, replace=False)
x_t_pick  = val_ds.X_t[picks].to(device)
x_t1_pick = val_ds.X_t1[picks].to(device)
with torch.no_grad():
    x_t1_hat = dec(enc(x_t_pick)).cpu().numpy()
x_t_np  = x_t_pick.cpu().numpy()
x_t1_np = x_t1_pick.cpu().numpy()

olr_ch = 2
fig, axes = plt.subplots(4, 3, figsize=(13, 14), sharex=True, sharey=True)
fig.suptitle("lp25 val-set reconstructions (OLR channel): X_t (left) | X_t+1 target (middle) | X̂_t+1 predicted (right)",
             fontsize=12, fontweight='bold')
vmax = max(np.abs(x_t_np[:, olr_ch]).max(), np.abs(x_t1_np[:, olr_ch]).max(), np.abs(x_t1_hat[:, olr_ch]).max())
for r in range(4):
    for c, (arr, label) in enumerate([(x_t_np, 'X_t'), (x_t1_np, 'X_t+1 target'), (x_t1_hat, 'X̂_t+1 predicted')]):
        ax = axes[r, c]
        im = ax.imshow(arr[r, olr_ch], cmap='RdBu_r', aspect='auto',
                       extent=[60, 160, 0, 60], vmin=-vmax, vmax=vmax, origin='lower')
        if r == 0: ax.set_title(label, fontsize=11, fontweight='bold')
        if c == 0: ax.set_ylabel(f'pair {picks[r]}', fontsize=10)
        if r == 3: ax.set_xlabel('Longitude (°)')
fig.subplots_adjust(right=0.90)
cb = fig.add_axes([0.92, 0.15, 0.015, 0.7])
plt.colorbar(im, cax=cb, label="OLR' (σ)")
plt.savefig(f'{RESULTS_DIR}/reconstructions.png', dpi=130, bbox_inches='tight')
plt.show()

# 3) Latent diagnostics — per-dim std bar chart + a quick scree check
with torch.no_grad():
    z_train_diag = np.concatenate([enc(train_ds.X_t[k:k+256].to(device)).cpu().numpy()
                                    for k in range(0, len(train_ds), 256)], axis=0)
z_std  = z_train_diag.std(axis=0)
z_mean = z_train_diag.mean(axis=0)
n_active = int((z_std > 0.01).sum())

# Quick scree to compare against nb18's flat scree
from sklearn.decomposition import PCA
pca_diag = PCA(n_components=min(15, LATENT_DIM)).fit(z_train_diag - z_train_diag.mean(0))
var_ratio_diag = pca_diag.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
ax = axes[0]
ax.bar(np.arange(LATENT_DIM), z_std, color='steelblue', alpha=0.85)
ax.axhline(z_std.mean(), color='red', ls='--', lw=1, label=f'Mean std = {z_std.mean():.3f}')
ax.axhline(0.01, color='gray', ls=':', lw=1, label='Collapse threshold (0.01)')
ax.set_xlabel('Latent dim'); ax.set_ylabel('std across train pairs')
ax.set_title(f'Per-Dim std  (active: {n_active}/{LATENT_DIM}; nb18 was 64/64 with mean ≈ 0.031)',
             fontweight='bold', fontsize=11)
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.bar(range(1, len(var_ratio_diag)+1), var_ratio_diag*100, color='steelblue', alpha=0.85)
ax.set_xlabel('PC index'); ax.set_ylabel('% variance')
ax.set_title(f'PCA scree (first {len(var_ratio_diag)} PCs)\n'
             f'PC1 = {var_ratio_diag[0]*100:.1f}%  (nb18 was 6.5% — flat scree = no manifold structure)',
             fontweight='bold', fontsize=11)
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/latent_diagnostics.png', dpi=140, bbox_inches='tight')
plt.show()

print(f'\nz_train latent summary (lp25):')
print(f'  active dims:  {n_active}/{LATENT_DIM}')
print(f'  std per dim:  min={z_std.min():.4f}, max={z_std.max():.4f}, mean={z_std.mean():.4f}')
print(f'  PC1 / PC2:    {var_ratio_diag[0]*100:.2f}%  /  {var_ratio_diag[1]*100:.2f}%')
print(f'  cum at 5 PCs: {np.cumsum(var_ratio_diag)[4]*100:.2f}%')
if var_ratio_diag[0] > 0.20:
    print('✓ PC1 dominant — manifold structure looks healthier than nb18.')
else:
    print('△ PC1 still small — manifold may still be diffuse.')

## Cell 6 — Extract and Save Latent Vectors for nb19

In [ ]:
def extract_z(dataset, encoder, batch=256):
    encoder.eval()
    zs = []
    with torch.no_grad():
        for k in range(0, len(dataset), batch):
            zs.append(encoder(dataset.X_t[k:k+batch].to(device)).cpu().numpy())
    return np.concatenate(zs, axis=0).astype(np.float32)

z_train = extract_z(train_ds, enc)
z_val   = extract_z(val_ds,   enc)
np.save(f'{LATENT_DIR}/z_train.npy', z_train)
np.save(f'{LATENT_DIR}/z_val.npy',   z_val)

print(f'Saved lp25 latents:')
print(f'  z_train.npy  shape {z_train.shape}  range [{z_train.min():.3f}, {z_train.max():.3f}]')
print(f'  z_val.npy    shape {z_val.shape}    range [{z_val.min():.3f}, {z_val.max():.3f}]')

import time as _t
stage1_summary = {
    'date':                _t.strftime('%Y-%m-%d'),
    'variant':             'lp25 (lowpassed input; addresses nb18 synoptic-noise confound)',
    'latent_dim':          LATENT_DIM,
    'epochs':              EPOCHS,
    'best_val_mse':        float(best_val),
    'final_train_mse':     float(history['train_mse'][-1]),
    'final_val_mse':       float(history['val_mse'][-1]),
    'persistence_mse':     float(persistence_mse),
    'improvement_pct':     float((persistence_mse - best_val) / persistence_mse * 100),
    'n_active_dims':       n_active,
    'pc1_var_ratio':       float(var_ratio_diag[0]),
    'pc2_var_ratio':       float(var_ratio_diag[1]),
    'cum_var_5pcs':        float(np.cumsum(var_ratio_diag)[4]),
    'z_train_shape':       list(z_train.shape),
    'z_val_shape':         list(z_val.shape),
    'meta_from_nb17b':     meta,
}
with open(f'{RESULTS_DIR}/stage1_summary.json', 'w') as f:
    json.dump(stage1_summary, f, indent=2)
print(f'\nSaved summary: {RESULTS_DIR}/stage1_summary.json')
print('\n✓ Stage 1 on lp25 complete.')
print('\nTo run nb19 on these results: at the top of nb19 Cell 1, change')
print(f'  LATENT_DIR  = f\'{{NSV_DIR}}/latents_lp25\'')
print(f'  RESULTS_DIR = f\'{{NSV_DIR}}/results/stage2_lp25\'')
print('and re-run.')

---
## Done!

**Send back** for review:
1. The Cell 4 final printed lines (best/final MSE + persistence + improvement %).
2. `results/stage1_lp25/training_curves.png`.
3. `results/stage1_lp25/reconstructions.png` — the predictions should look like smooth, large-scale BSISO patterns. Less blurry than nb18's because lp25 targets are themselves smoother.
4. `results/stage1_lp25/latent_diagnostics.png` — **the key panel** is the PC1 % on the right; if PC1 jumps from nb18's 6.5% to 30%+, the manifold has structure and nb19 will give a meaningful ID.
5. `results/stage1_lp25/stage1_summary.json` — top-line numbers.

**Then re-run nb19** with the two path changes printed above. Expected result: `d̂` in the 2–5 range with HIGH or MEDIUM confidence, the PCA scatter showing visible BSISO phase organization, the noise control still saturating at ~40 (sample-size limit) but the real-data ID well below it.

---
*DDCS Project | jh9141@nyu.edu*